# DermaScope — out-of-distribution gate

Builds a gate that rejects non-dermatoscopic images before the classifier runs.

1. Export a float32 TFLite with two outputs — class probabilities and the 1280-dimensional
   pooled features — and confirm its probabilities match `skin_cancer_model_f32.tflite`
   (the model the app loads) on the full test set.
2. Rebuild the training split exactly as the training notebook does (scikit-learn 1.7.2),
   embed up to 3,000 training images with the app's exact preprocessing, L2-normalize, and
   save them as `ood_bank.npy` (float16).
3. Score an image by its cosine distance to its k-th nearest neighbor in the bank, for
   k = 5, 10 and 50. For each k the threshold is the 95th percentile of validation-set
   scores. No negative image is used to set a threshold.
4. For each k, report the rejection rate on the test set per lesion class, on general photos
   (easy negatives: people, animals, objects) and on DermNet clinical skin photos (hard
   negatives). Pick one k and save its threshold to `ood_threshold.json`.

In [ ]:
# QUICK_TEST = True limits every group of images (test, bank, validation, general
# photos, DermNet) to 50, to check that every cell runs. Its numbers mean nothing.
QUICK_TEST  = False
QUICK_LIMIT = 50

import time

RUN_START = time.time()
_cell = {'name': None, 'start': None, 'count': 0}

def _clock():
    elapsed = time.time() - RUN_START
    return f"+{int(elapsed // 60):02d}:{int(elapsed % 60):02d}"

def cell_start(name):
    _cell.update(name=name, start=time.time(), count=_cell['count'] + 1)
    print(f"▶ cell {_cell['count']} [{name}] started at {_clock()}", flush=True)

def cell_end():
    print(f"■ cell {_cell['count']} [{_cell['name']}] finished in {time.time() - _cell['start']:.1f}s "
          f"(run {_clock()})", flush=True)

def progress(label, done, total, start, every=200):
    """Print every `every` images, and at the end."""
    if done % every == 0 or done == total:
        print(f"    {label}: {done}/{total} images, {time.time() - start:.0f}s", flush=True)

cell_start("setup")
print("QUICK_TEST =", QUICK_TEST, "(every group limited to 50 images)" if QUICK_TEST else "(full run)")

# The training split only reproduces on some scikit-learn versions, so pin it
# before anything imports sklearn.
!pip install -q "scikit-learn==1.7.2"

cell_end()

In [ ]:
cell_start("imports")

import os
import json
import time
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import sklearn
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Model
from PIL import Image

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score

assert sklearn.__version__ == '1.7.2', f"Expected scikit-learn 1.7.2, got {sklearn.__version__}"

print("scikit-learn:", sklearn.__version__)
print("TensorFlow:  ", tf.__version__)
print("Keras:       ", keras.__version__)
print("pandas:      ", pd.__version__, "| numpy:", np.__version__, "| cv2:", cv2.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("GPU:         ", gpus[0].name if gpus else "none (not needed: TFLite inference runs on the CPU)")

cell_end()

In [ ]:
cell_start("config and inputs")

# Same settings as the training notebook's Config
IMG_SIZE    = 224
RANDOM_SEED = 42
N_FOLDS     = 20
OUT_DIR     = "/kaggle/working"

CLASSES = {
    'mel':   'Melanoma',
    'nv':    'Melanocytic Nevi',
    'bcc':   'Basal Cell Carcinoma',
    'akiec': 'Actinic Keratosis',
    'bkl':   'Benign Keratosis',
    'df':    'Dermatofibroma',
    'vasc':  'Vascular Lesion'
}
NUM_CLASSES = len(CLASSES)

# Gate settings
FEATURE_DIM      = 1280   # EfficientNetB0 pooled features
BANK_MAX         = 3000   # at most this many training images in the bank
K_CANDIDATES     = (5, 10, 50)   # score = cosine distance to the k-th nearest bank image
PERCENTILE       = 95     # threshold = this percentile of validation scores
NEG_PER_CATEGORY = 100    # general photos (easy negatives) sampled per category
HARD_NEG_COUNT   = 400    # DermNet clinical skin photos (hard negatives) sampled
# A k is only eligible if it still rejects at least this share of the general photos
EASY_NEG_MIN_REJECTION = 0.90

# Numbers logged by the training and verification runs, used as pass/fail targets below
LOGGED_SPLIT_SIZES  = (7003, 1528, 1484)                # train / val / test
LOGGED_TEST_COUNTS  = [162, 999, 53, 41, 169, 30, 30]
LOGGED_APP_ACCURACY = 0.7796   # float32 TFLite with the app's preprocessing, full test set

# SHA-256 of the model files committed in the GitHub repo
REPO_SHA256 = {
    'best_model.keras':             '2bf6c946860b2f50d9168d7f17d3f43fedbe79fe3b717c32563e43d6b0b7d9b9',
    'skin_cancer_model_f32.tflite': '56b733b9a331a7d8e1208616a232830109393884d5f9744e63ee46e78762816b',
}

# Locate inputs by filename so this works whatever the mount layout is
all_files = sorted(os.path.join(d, f) for d, _, fs in os.walk('/kaggle/input') for f in fs)

def find_input(filename):
    hits = [p for p in all_files if os.path.basename(p) == filename]
    assert hits, f"{filename} not found under /kaggle/input"
    if len(hits) > 1:
        print(f"  note: {len(hits)} copies of {filename}, using the first")
    return hits[0]

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

METADATA_PATH = find_input('HAM10000_metadata.csv')
KERAS_PATH    = find_input('best_model.keras')

for p in (METADATA_PATH, KERAS_PATH):
    print(p)
assert sha256(KERAS_PATH) == REPO_SHA256['best_model.keras'], \
    "best_model.keras is not the file in the repo — stop here."
print("best_model.keras: identical to the file in the repo")

IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png')

def quick_limit(dataframe):
    """In a quick test, keep a fixed random 50 rows; otherwise keep everything."""
    if QUICK_TEST and len(dataframe) > QUICK_LIMIT:
        return dataframe.sample(QUICK_LIMIT, random_state=RANDOM_SEED).sort_index()
    return dataframe

# Easy negatives: prasunroy/natural-images, one folder per category.
# The dataset ships the same tree twice, so keep one copy of each file.
neg_files = {}
for p in all_files:
    parts = p.split(os.sep)
    if 'natural_images' in parts[:-2] and p.lower().endswith(IMAGE_EXTENSIONS):
        neg_files.setdefault((parts[-2], parts[-1]), p)
neg_all = pd.DataFrame([{'category': c, 'file': f, 'path': p} for (c, f), p in sorted(neg_files.items())])
assert len(neg_all), "No general photos found — is prasunroy/natural-images attached?"
print(f"\nGeneral photos available: {len(neg_all)}")
print(neg_all['category'].value_counts().sort_index().to_string())

# Hard negatives: shubhamgoel27/dermnet, clinical (non-dermatoscopic) skin photos,
# train/ and test/ folders with one folder per disease group.
hard_files = {}
for p in all_files:
    parts = p.split(os.sep)
    if 'dermnet' in [x.lower() for x in parts[:-2]] and p.lower().endswith(IMAGE_EXTENSIONS):
        hard_files.setdefault((parts[-3], parts[-2], parts[-1]), p)
hard_all = pd.DataFrame([{'category': c, 'file': f"{s}/{c}/{f}", 'path': p}
                         for (s, c, f), p in sorted(hard_files.items())])
assert len(hard_all), "No DermNet photos found — is shubhamgoel27/dermnet attached?"
print(f"\nDermNet photos available: {len(hard_all)} in {hard_all['category'].nunique()} disease folders")

cell_end()

## 1 — Two-output float32 TFLite

The reference is `skin_cancer_model_f32.tflite`, the model the app loads. It is re-exported
here from `best_model.keras` and must be byte-identical (SHA-256) to the file in the repo
before it is used as the reference. The two-output model is the same graph with one more
output: the `GlobalAveragePooling2D` layer, before the classification head.

In [ ]:
cell_start("export two-output TFLite")

best_model = keras.models.load_model(KERAS_PATH)

# Reference: the single-output float32 export the app loads.
# No converter.optimizations → weights stay float32.
APP_TFLITE_PATH = f"{OUT_DIR}/skin_cancer_model_f32.tflite"
with open(APP_TFLITE_PATH, 'wb') as f:
    f.write(tf.lite.TFLiteConverter.from_keras_model(best_model).convert())
assert sha256(APP_TFLITE_PATH) == REPO_SHA256['skin_cancer_model_f32.tflite'], \
    "Re-exported skin_cancer_model_f32.tflite is not the file the app loads — stop here."
print("skin_cancer_model_f32.tflite: identical to the file the app loads")

# Two outputs: class probabilities and the pooled features. EfficientNet also
# has a pooling layer inside every squeeze-and-excite block, so pick the one
# that pools the final 7x7x1280 feature map and feeds the classification head.
pool_layers = [l for l in best_model.layers
               if isinstance(l, keras.layers.GlobalAveragePooling2D) and l.output.shape[-1] == FEATURE_DIM]
assert len(pool_layers) == 1, f"Expected one {FEATURE_DIM}-dimensional pooling layer, found {len(pool_layers)}"
pool_layer = pool_layers[0]
assert tuple(pool_layer.input.shape[1:]) == (7, 7, FEATURE_DIM), pool_layer.input.shape
assert pool_layer is [l for l in best_model.layers if isinstance(l, keras.layers.GlobalAveragePooling2D)][-1]
print(f"Pooled-feature layer: {pool_layer.name} {tuple(pool_layer.output.shape)}")

gate_model = Model(best_model.inputs, [best_model.output, pool_layer.output])

GATE_TFLITE_PATH = f"{OUT_DIR}/skin_cancer_model_f32_features.tflite"
with open(GATE_TFLITE_PATH, 'wb') as f:
    f.write(tf.lite.TFLiteConverter.from_keras_model(gate_model).convert())
print(f"Saved {GATE_TFLITE_PATH} ({os.path.getsize(GATE_TFLITE_PATH) / 1e6:.1f} MB)")

cell_end()

In [ ]:
cell_start("interpreters and app preprocessing")

try:
    from ai_edge_litert.interpreter import Interpreter   # what the app uses
    print("Interpreter: ai_edge_litert")
except ImportError:
    Interpreter = tf.lite.Interpreter
    print("Interpreter: tf.lite (ai_edge_litert not installed here)")


# ── Copied from dermascope/skin_cancer_app.py @ 817ba13 ──
def preprocess_image(img: Image.Image) -> np.ndarray:
    """PIL Image → float32 array in [0, 255], resized the way the training pipeline does."""
    arr = np.array(img.convert("RGB"), dtype=np.uint8)
    arr = cv2.resize(arr, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
    arr = arr.astype(np.float32)
    return np.expand_dims(arr, axis=0)


def run_inference(interpreter, input_data: np.ndarray) -> np.ndarray:
    """Run TFLite inference. Returns probability array of shape (7,)."""
    input_details  = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details[0]['index'])
    return output[0]
# ── End copied code ──


def load_gate_interpreter(model_path):
    """Load the two-output model. The converter does not promise an output
    order, so the two outputs are told apart by their size."""
    interpreter = Interpreter(model_path=model_path)
    interpreter.allocate_tensors()
    by_size = {int(d['shape'][-1]): d['index'] for d in interpreter.get_output_details()}
    assert set(by_size) == {NUM_CLASSES, FEATURE_DIM}, f"Unexpected outputs: {by_size}"
    return interpreter, by_size[NUM_CLASSES], by_size[FEATURE_DIM]


def run_gate_inference(gate, input_data: np.ndarray):
    """One image through the two-output model. Returns (probabilities (7,), features (1280,))."""
    interpreter, probs_index, features_index = gate
    interpreter.set_tensor(interpreter.get_input_details()[0]['index'], input_data)
    interpreter.invoke()
    return (interpreter.get_tensor(probs_index)[0].copy(),
            interpreter.get_tensor(features_index)[0].copy())


gate = load_gate_interpreter(GATE_TFLITE_PATH)
for d in gate[0].get_output_details():
    print("  output:", d['name'], tuple(d['shape']), d['dtype'].__name__)


def embed_images(paths, reference=None, label=""):
    """Run every image through the app's preprocessing and the two-output model.
    If a reference interpreter is given, the same input also goes through the
    app's own run_inference() so the two sets of probabilities can be compared."""
    n = len(paths)
    probs     = np.empty((n, NUM_CLASSES), dtype=np.float32)
    features  = np.empty((n, FEATURE_DIM), dtype=np.float32)
    ref_probs = np.empty((n, NUM_CLASSES), dtype=np.float32) if reference is not None else None
    start = time.time()
    for i, path in enumerate(paths):
        # Same steps as the app: open → RGB → preprocess → infer
        input_data = preprocess_image(Image.open(path).convert("RGB"))
        probs[i], features[i] = run_gate_inference(gate, input_data)
        if reference is not None:
            ref_probs[i] = run_inference(reference, input_data)
        progress(label, i + 1, n, start)
    return probs, features, ref_probs

cell_end()

## 2 — Rebuild the split

In [ ]:
cell_start("rebuild split")

df = pd.read_csv(METADATA_PATH)

# Build full image paths (the dataset ships the images in more than one
# folder; duplicates are the same files, so the first hit is fine)
all_image_paths = {}
for p in all_files:
    if p.endswith('.jpg') and os.path.basename(p).startswith('ISIC_'):
        all_image_paths.setdefault(os.path.splitext(os.path.basename(p))[0], p)

df['path'] = df['image_id'].map(all_image_paths)
print(f"Images found: {df['path'].notnull().sum()} / {len(df)}")
df = df.dropna(subset=['path'])

label_to_idx = {dx: i for i, dx in enumerate(CLASSES.keys())}
idx_to_label = {i: dx for dx, i in label_to_idx.items()}
df['label_idx'] = df['dx'].map(label_to_idx)

# Identical to the training notebook: 20 lesion-grouped stratified folds,
# 14 → train, 3 → val, 3 → test.
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
fold_indices = [held_out_idx for _, held_out_idx in
                sgkf.split(df, df['label_idx'], groups=df['lesion_id'])]

train_idx = np.concatenate(fold_indices[0:14])
val_idx   = np.concatenate(fold_indices[14:17])
test_idx  = np.concatenate(fold_indices[17:20])

train_df = df.iloc[train_idx]
val_df   = df.iloc[val_idx]
test_df  = df.iloc[test_idx]
print(f"Train {len(train_df)} / Val {len(val_df)} / Test {len(test_df)}")

train_lesions, val_lesions, test_lesions = (set(d['lesion_id']) for d in (train_df, val_df, test_df))
assert not (train_lesions & val_lesions or train_lesions & test_lesions or val_lesions & test_lesions), \
    "Lesion overlap between splits!"

test_counts = test_df['label_idx'].value_counts().reindex(range(NUM_CLASSES), fill_value=0).tolist()
print("Test per-class counts:", test_counts)
print("Logged by training:   ", LOGGED_TEST_COUNTS)
assert (len(train_df), len(val_df), len(test_df)) == LOGGED_SPLIT_SIZES, \
    "Split sizes do not match the training run — stop here."
assert test_counts == LOGGED_TEST_COUNTS, "Test split does not match the training run — stop here."
print("✓ Split matches the training run (sizes and test per-class counts)")

# Quick test only: the split above is still built and checked in full
val_df, test_df = quick_limit(val_df), quick_limit(test_df)
if QUICK_TEST:
    print(f"QUICK_TEST: using {len(val_df)} validation and {len(test_df)} test images")

cell_end()

### Probabilities match the app's model on the full test set

Every test image goes through the app's preprocessing once, and that same input is fed to
both models: the app's `skin_cancer_model_f32.tflite` through the app's own `run_inference`,
and the two-output model.

In [ ]:
cell_start("match app model on test set")

MATCH_TOLERANCE = 1e-4   # largest allowed difference in any class probability

reference = Interpreter(model_path=APP_TFLITE_PATH)
reference.allocate_tensors()

test_probs, test_features, test_ref_probs = embed_images(test_df['path'].values, reference, "test set, both models")

max_diff  = float(np.abs(test_probs - test_ref_probs).max())
agreement = float(np.mean(test_probs.argmax(axis=1) == test_ref_probs.argmax(axis=1)))
acc_gate  = accuracy_score(test_df['label_idx'].values, test_probs.argmax(axis=1))
acc_ref   = accuracy_score(test_df['label_idx'].values, test_ref_probs.argmax(axis=1))

print("=" * 60)
print(f"TWO-OUTPUT MODEL vs skin_cancer_model_f32.tflite — n = {len(test_df)}")
print("=" * 60)
print(f"Largest probability difference: {max_diff:.2e}")
print(f"Bit-identical probabilities:    {np.array_equal(test_probs, test_ref_probs)}")
print(f"Same predicted class:           {agreement:.4%}")
print(f"Accuracy: two-output {acc_gate:.4f} | app model {acc_ref:.4f} | logged {LOGGED_APP_ACCURACY:.4f}")

assert max_diff <= MATCH_TOLERANCE and agreement == 1.0, \
    "Two-output model does not match the app's model — stop here."
if not QUICK_TEST:   # the logged accuracy is for the full test set
    assert abs(acc_ref - LOGGED_APP_ACCURACY) < 0.0005, "App model accuracy does not match the verified 77.96%"
print("✓ Probabilities match the model the app loads")

cell_end()

## 3 — Feature bank

Every non-nevus training image, plus a random sample of nevi to bring the total to 3,000.
Nevi are two thirds of the training split, so this keeps all of the rarer classes.

In [ ]:
cell_start("feature bank")

non_nevus = train_df[train_df['dx'] != 'nv']
nevi      = train_df[train_df['dx'] == 'nv']
assert len(non_nevus) <= BANK_MAX, f"{len(non_nevus)} non-nevus training images do not fit in a bank of {BANK_MAX}"

n_nevi  = min(len(nevi), BANK_MAX - len(non_nevus))
bank_df = pd.concat([non_nevus, nevi.sample(n_nevi, random_state=RANDOM_SEED)]).sort_values('image_id')
assert len(bank_df) <= BANK_MAX and set(bank_df['lesion_id']) <= train_lesions
bank_df = quick_limit(bank_df)

print(f"Bank: {len(bank_df)} of {len(train_df)} training images "
      f"({len(non_nevus)} non-nevus + {n_nevi} of {len(nevi)} nevi)")
bank_counts = pd.DataFrame({
    'in_training_split': train_df['dx'].value_counts(),
    'in_bank':           bank_df['dx'].value_counts(),
}).reindex(list(CLASSES.keys()))
print(bank_counts.to_string())

_, bank_features, _ = embed_images(bank_df['path'].values, label="bank")


def l2_normalize(features):
    return features / np.linalg.norm(features, axis=-1, keepdims=True)


BANK_PATH = f"{OUT_DIR}/ood_bank.npy"
np.save(BANK_PATH, l2_normalize(bank_features).astype(np.float16))
bank_df[['image_id', 'lesion_id', 'dx']].to_csv(f"{OUT_DIR}/ood_bank_images.csv", index=False)

# Everything below uses the bank as it was saved, the way the app will load it
bank = np.load(BANK_PATH)
print(f"\nSaved {BANK_PATH}: shape {bank.shape}, dtype {bank.dtype}, "
      f"{os.path.getsize(BANK_PATH) / 1e6:.1f} MB")
bank = bank.astype(np.float32)
norms = np.linalg.norm(bank, axis=1)
print(f"Row norms after float16 rounding: {norms.min():.5f} – {norms.max():.5f}")

cell_end()

## 4 — Scores

Score = cosine distance from the image's L2-normalized features to its k-th nearest neighbor
in the bank. A higher score means the image is further from the training images.
`ood_score` is the single-image function the app will use; the sweep below computes the same
thing for whole sets at once and is checked against it.

In [ ]:
cell_start("embed and score")

def ood_score(features, bank, k):
    """Cosine distance to the k-th nearest bank image. `features` is one image's
    pooled features (1280,); `bank` is the saved bank as float32."""
    query = features / np.linalg.norm(features)
    similarities = bank @ query
    kth_largest = np.partition(similarities, -k)[-k]
    return float(1.0 - kth_largest)


def ood_scores_all_k(features):
    """Scores for every k in K_CANDIDATES, as {k: array of one score per image}."""
    assert max(K_CANDIDATES) <= len(bank), f"Bank of {len(bank)} is smaller than k = {max(K_CANDIDATES)}"
    similarities = l2_normalize(features) @ bank.T
    top = -np.sort(-similarities, axis=1)[:, :max(K_CANDIDATES)]   # largest first
    scores = {k: 1.0 - top[:, k - 1] for k in K_CANDIDATES}
    for k in K_CANDIDATES:   # same numbers as the single-image function
        single = [ood_score(f, bank, k) for f in features[:25]]
        assert np.allclose(scores[k][:25], single, atol=1e-6)
    return scores


def keep_readable(dataframe, label):
    """Drop images that cannot be opened, so one bad file does not stop the run."""
    keep, start = [], time.time()
    for i, path in enumerate(dataframe['path'].values):
        try:
            Image.open(path).convert("RGB")
            keep.append(True)
        except Exception:
            keep.append(False)
        progress(f"{label}, readable check", i + 1, len(dataframe), start)
    return dataframe[keep].reset_index(drop=True)


MIN_NEGATIVES = 1 if QUICK_TEST else 200

# Easy negatives: the same number of general photos from every category
neg_df = pd.concat([g.sample(min(len(g), NEG_PER_CATEGORY), random_state=RANDOM_SEED)
                    for _, g in neg_all.groupby('category')]).reset_index(drop=True)
neg_df = keep_readable(quick_limit(neg_df), "general photos")
assert len(neg_df) >= MIN_NEGATIVES, f"Only {len(neg_df)} general photos — need at least {MIN_NEGATIVES}"

# Hard negatives: a random sample of DermNet clinical photos
hard_df = hard_all.sample(min(len(hard_all), HARD_NEG_COUNT), random_state=RANDOM_SEED).reset_index(drop=True)
hard_df = keep_readable(quick_limit(hard_df), "DermNet photos")
assert len(hard_df) >= MIN_NEGATIVES, f"Only {len(hard_df)} DermNet photos — need at least {MIN_NEGATIVES}"

_, val_features, _           = embed_images(val_df['path'].values, label="validation set")
neg_probs, neg_features, _   = embed_images(neg_df['path'].values, label="general photos")
hard_probs, hard_features, _ = embed_images(hard_df['path'].values, label="DermNet photos")

val_scores  = ood_scores_all_k(val_features)
test_scores = ood_scores_all_k(test_features)
neg_scores  = ood_scores_all_k(neg_features)
hard_scores = ood_scores_all_k(hard_features)
print("✓ Set-wide scores match the single-image ood_score() for every k")

cell_end()

## 5 — k = 5, 10, 50

For each k the threshold is the 95th percentile of that k's validation scores — so about 5%
of in-distribution images are rejected by design. An image is rejected when its score is
above the threshold. Only validation images set the threshold.

In [ ]:
cell_start("k sweep")

thresholds = {k: float(np.percentile(val_scores[k], PERCENTILE)) for k in K_CANDIDATES}

sweep_rows = []

def add_rows(k, split, groups, rejected):
    """One row for the whole split and one per group."""
    sweep_rows.append({'k': k, 'split': split, 'group': 'all', 'n': len(rejected),
                       'rejected': int(rejected.sum()), 'rejection_rate': float(rejected.mean())})
    if groups is not None:
        for group in pd.unique(groups):
            mask = np.asarray(groups == group)
            sweep_rows.append({'k': k, 'split': split, 'group': group, 'n': int(mask.sum()),
                               'rejected': int(rejected[mask].sum()),
                               'rejection_rate': float(rejected[mask].mean())})

for k in K_CANDIDATES:
    add_rows(k, 'validation',     None,                      val_scores[k]  > thresholds[k])
    add_rows(k, 'test',           test_df['dx'].values,      test_scores[k] > thresholds[k])
    add_rows(k, 'general_photos', neg_df['category'].values, neg_scores[k]  > thresholds[k])
    add_rows(k, 'dermnet',        None,                      hard_scores[k] > thresholds[k])

sweep = pd.DataFrame(sweep_rows)

def rate_table(split):
    rows = sweep[sweep['split'] == split]
    table = rows.pivot(index='group', columns='k', values='rejection_rate')
    table.columns = [f"k={k}" for k in table.columns]
    table.insert(0, 'n', rows.drop_duplicates('group').set_index('group')['n'])
    return table

print("Thresholds:", {k: round(t, 6) for k, t in thresholds.items()})

print("\n" + "=" * 60)
print(f"TEST SET — rejection rate by lesion class (n = {len(test_df)})")
print("=" * 60)
print(rate_table('test').reindex(list(CLASSES.keys()) + ['all']).round(4).to_string())

print("\n" + "=" * 60)
print(f"GENERAL PHOTOS (easy negatives) — rejection rate by category (n = {len(neg_df)})")
print("=" * 60)
print(rate_table('general_photos').round(4).to_string())

print("\n" + "=" * 60)
print(f"DERMNET CLINICAL PHOTOS (hard negatives) — rejection rate (n = {len(hard_df)})")
print("=" * 60)
print(rate_table('dermnet').round(4).to_string())

cell_end()

### Choosing k

Rule: among the values of k that still reject at least 90% of the general photos, take the one
whose worst lesion class has the lowest test-set rejection rate. Ties go to the k that rejects
more DermNet photos, then more general photos, then the larger k.

The choice is made on the test set, so the chosen k's test-set numbers are slightly optimistic.

In [ ]:
cell_start("choose k")

def overall(k, split):
    return float(sweep[(sweep['k'] == k) & (sweep['split'] == split) & (sweep['group'] == 'all')]['rejection_rate'].iloc[0])

summary_rows = []
for k in K_CANDIDATES:
    by_class = sweep[(sweep['k'] == k) & (sweep['split'] == 'test') & (sweep['group'] != 'all')]
    worst = by_class.sort_values('rejection_rate', ascending=False).iloc[0]
    summary_rows.append({
        'k': k,
        'threshold': thresholds[k],
        'test_rejection': overall(k, 'test'),
        'worst_class': worst['group'],
        'worst_class_rejection': float(worst['rejection_rate']),
        'general_photos_rejection': overall(k, 'general_photos'),
        'dermnet_rejection': overall(k, 'dermnet'),
    })
summary = pd.DataFrame(summary_rows)
summary['eligible'] = summary['general_photos_rejection'] >= EASY_NEG_MIN_REJECTION
print(summary.round(4).to_string(index=False))

eligible = summary[summary['eligible']]
if QUICK_TEST and not len(eligible):
    print("QUICK_TEST: no k is eligible on 50 images; ranking all of them so the remaining cells run")
    eligible = summary
assert len(eligible), f"No k rejects at least {EASY_NEG_MIN_REJECTION:.0%} of the general photos — stop here."

ranked = eligible.sort_values(
    ['worst_class_rejection', 'dermnet_rejection', 'general_photos_rejection', 'k'],
    ascending=[True, False, False, False])
choice    = ranked.iloc[0]
K_CHOSEN  = int(choice['k'])
THRESHOLD = thresholds[K_CHOSEN]

print("\n" + "=" * 60)
print(f"CHOSEN: k = {K_CHOSEN}, threshold {THRESHOLD:.6f}")
print("=" * 60)
for _, row in summary.iterrows():
    note = "" if row['eligible'] else f"  — not eligible (general photos below {EASY_NEG_MIN_REJECTION:.0%})"
    print(f"k = {int(row['k']):>2}: worst lesion class {row['worst_class']} at {row['worst_class_rejection']:.1%} | "
          f"general photos {row['general_photos_rejection']:.1%} | DermNet {row['dermnet_rejection']:.1%}{note}")
tied = ranked[np.isclose(ranked['worst_class_rejection'], choice['worst_class_rejection'])]
if len(tied) > 1:
    print(f"k = {', '.join(str(int(k)) for k in tied['k'])} tie on the worst lesion class; "
          f"k = {K_CHOSEN} wins the tie-break (DermNet, then general photos, then larger k).")
else:
    print(f"k = {K_CHOSEN} has the lowest worst-case rejection rate across lesion classes among the eligible values.")

threshold_info = {
    'quick_test':       QUICK_TEST,   # True → 50 images per group, do not use this file
    'threshold':        THRESHOLD,
    'rule':             'reject when score > threshold',
    'score':            f'cosine distance to the k-th nearest neighbor in the bank, k = {K_CHOSEN}',
    'k':                K_CHOSEN,
    'percentile':       PERCENTILE,
    'n_validation':     int(len(val_df)),
    'bank_file':        'ood_bank.npy',
    'bank_size':        int(bank.shape[0]),
    'bank_dtype':       'float16',
    'feature_dim':      FEATURE_DIM,
    'features':         'GlobalAveragePooling2D output, L2-normalized',
    'model_file':       os.path.basename(GATE_TFLITE_PATH),
    'model_sha256':     sha256(GATE_TFLITE_PATH),
    'bank_sha256':      sha256(BANK_PATH),
    'k_selection':      f'lowest worst-case test rejection rate across lesion classes, among k that reject '
                        f'at least {EASY_NEG_MIN_REJECTION:.0%} of general photos',
    'k_candidates':     json.loads(summary.to_json(orient='records')),
}
with open(f"{OUT_DIR}/ood_threshold.json", 'w') as f:
    json.dump(threshold_info, f, indent=2)
print("\n" + json.dumps(threshold_info, indent=2))

cell_end()

## 6 — The chosen k in detail

In [ ]:
cell_start("chosen k in detail")

def detail(frame, scores, probs=None):
    out = frame.copy()
    for k in K_CANDIDATES:
        out[f'score_k{k}'] = scores[k]
    out['rejected'] = scores[K_CHOSEN] > THRESHOLD
    if probs is not None:   # what the classifier would have said without a gate
        out['classifier_top_class']  = [idx_to_label[i] for i in probs.argmax(axis=1)]
        out['classifier_confidence'] = probs.max(axis=1)
    return out

score_col = f'score_k{K_CHOSEN}'
val_out  = detail(val_df[['image_id', 'dx']], val_scores)
test_out = detail(test_df[['image_id', 'dx']], test_scores)
neg_out  = detail(neg_df[['category', 'file']], neg_scores, neg_probs)
hard_out = detail(hard_df[['category', 'file']], hard_scores, hard_probs)

print("=" * 60)
print(f"TEST SET, k = {K_CHOSEN} — rejected {test_out['rejected'].sum()} / {len(test_out)} = "
      f"{test_out['rejected'].mean():.2%}  (expected: about {100 - PERCENTILE}%)")
print("=" * 60)
print(test_out.groupby('dx').agg(n=('rejected', 'size'), rejected=('rejected', 'sum'),
                                 rejection_rate=('rejected', 'mean'), median_score=(score_col, 'median'))
      .reindex(list(CLASSES.keys())).round(4).to_string())

for title, out in (("GENERAL PHOTOS (easy negatives)", neg_out), ("DERMNET CLINICAL PHOTOS (hard negatives)", hard_out)):
    print("\n" + "=" * 60)
    print(f"{title}, k = {K_CHOSEN} — rejected {out['rejected'].sum()} / {len(out)} = {out['rejected'].mean():.2%}")
    print("=" * 60)
    print(out.groupby('category').agg(n=('rejected', 'size'), rejected=('rejected', 'sum'),
                                      rejection_rate=('rejected', 'mean'), median_score=(score_col, 'median'),
                                      mean_classifier_confidence=('classifier_confidence', 'mean'))
          .round(4).to_string())
    accepted = out[~out['rejected']]
    print(f"\nLet through by the gate: {len(accepted)}. Of those, "
          f"{np.mean(accepted['classifier_confidence'] > 0.75) if len(accepted) else 0:.1%} "
          f"would get a 'high confidence' result from the classifier.")
    print(f"Without a gate: mean classifier confidence {out['classifier_confidence'].mean():.1%}; "
          f"{np.mean(out['classifier_confidence'] > 0.75):.1%} get a 'high confidence' result.")

cell_end()

In [ ]:
cell_start("save scores and plot")

# Save every score (all three k) and the full sweep
score_cols = [f'score_k{k}' for k in K_CANDIDATES]
scores_out = pd.concat([
    val_out.rename(columns={'image_id': 'image', 'dx': 'group'}).assign(split='validation'),
    test_out.rename(columns={'image_id': 'image', 'dx': 'group'}).assign(split='test'),
    neg_out.rename(columns={'file': 'image', 'category': 'group'}).assign(split='general_photos'),
    hard_out.rename(columns={'file': 'image', 'category': 'group'}).assign(split='dermnet'),
])[['split', 'group', 'image'] + score_cols + ['rejected']]
scores_out.to_csv(f"{OUT_DIR}/ood_scores.csv", index=False)

sweep.assign(threshold=sweep['k'].map(thresholds), chosen=sweep['k'] == K_CHOSEN) \
     .to_csv(f"{OUT_DIR}/ood_k_sweep.csv", index=False)

fig, axes = plt.subplots(1, len(K_CANDIDATES), figsize=(7 * len(K_CANDIDATES), 5))
for ax, k in zip(axes, K_CANDIDATES):
    top = max(s[k].max() for s in (val_scores, test_scores, neg_scores, hard_scores))
    bins = np.linspace(0, top * 1.02, 70)
    ax.hist(val_scores[k],  bins=bins, alpha=0.55, density=True, label=f'Validation (n={len(val_df)})')
    ax.hist(test_scores[k], bins=bins, alpha=0.55, density=True, label=f'Test (n={len(test_df)})')
    ax.hist(hard_scores[k], bins=bins, alpha=0.55, density=True, label=f'DermNet (n={len(hard_df)})')
    ax.hist(neg_scores[k],  bins=bins, alpha=0.55, density=True, label=f'General photos (n={len(neg_df)})')
    ax.axvline(thresholds[k], color='black', linestyle='--', label=f'Threshold {thresholds[k]:.4f}')
    ax.set_xlabel(f'Cosine distance to the {k}th-nearest bank image')
    ax.set_ylabel('Density')
    ax.set_title(f'k = {k}' + (' (chosen)' if k == K_CHOSEN else ''), fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/ood_score_distribution.png", dpi=150, bbox_inches='tight')
plt.show()

cell_end()

In [ ]:
cell_start("list outputs")

# The re-exported reference is already in the repo; no need to ship it again
os.remove(APP_TFLITE_PATH)

print("Files saved:")
for f in sorted(os.listdir(OUT_DIR)):
    print(f"  {OUT_DIR}/{f}  ({os.path.getsize(os.path.join(OUT_DIR, f)) / 1e6:.2f} MB)")

print(f"\nAll cells ran. Total run time {_clock()}.")
if QUICK_TEST:
    print("QUICK_TEST run: 50 images per group. The bank, threshold and rates above are NOT usable.")

cell_end()